# AI CV Screening Benchmark: Current vs. Proposed Structured Hybrid Architecture

This benchmark notebook evaluates whether incorporating **Structured Quantitative Rules** (GPA, degree hierarchy, experience duration) and **Qualitative LLM Experience Scoring** improves ranking correlation against recruiter ground-truth rankings sufficiently to justify the additional computational latency per CV compared to the **Current Vector Similarity** pipeline.

## 1. Imports, Device Detection, and Model Preloading

In [1]:
%pip install pymupdf pdf2image pytesseract langdetect deep-translator sentence-transformers spacy pandas scipy scikit-learn pydantic ollama --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [2]:
!apt-get update && apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ind

Reading package lists... Done
E: Could not open lock file /var/lib/apt/lists/lock - open (13: Permission denied)
E: Unable to lock directory /var/lib/apt/lists/
W: Problem unlinking the file /var/cache/apt/pkgcache.bin - RemoveCaches (13: Permission denied)
W: Problem unlinking the file /var/cache/apt/srcpkgcache.bin - RemoveCaches (13: Permission denied)


In [25]:
import os
import time
import json
import torch
import pandas as pd
import numpy as np
from scipy.stats import kendalltau, spearmanr
from ollama import Client

# Device Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# We will use Ollama for LLM tasks directly
ollama_client = Client(host="http://localhost:11434")

from sentence_transformers import SentenceTransformer
import spacy
from spacy.cli import download

print("Loading BGE-M3...")
bge_model = SentenceTransformer("BAAI/bge-m3", device=device)

print("Loading spaCy...")
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    print("Downloading spaCy model natively...")
    download("en_core_web_md")
    nlp = spacy.load("en_core_web_md")
    
print("Models loaded successfully!")

Device: cuda
Loading BGE-M3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24002.88it/s]


Loading spaCy...
Models loaded successfully!


In [26]:
import fitz
import pytesseract
from PIL import Image
import io
import os
from langdetect import detect
from deep_translator import GoogleTranslator

def extract_and_translate(pdf_path):
    text = ""
    try:
        # 1. Extract teks (pymupdf)
        doc = fitz.open(pdf_path)
        text = "\n".join([page.get_text() for page in doc])

        # 2. OCR Fallback (Bypasses Poppler completely)
        if len(text.strip()) < 150:
            print(f"  -> No text layer. Running native OCR on: {os.path.basename(pdf_path)}")
            ocr_text = []
            for page in doc:
                pix = page.get_pixmap(dpi=150)
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                try:
                    ocr_text.append(pytesseract.image_to_string(img))
                except Exception as e:
                    print(f"  -> Tesseract error: {e}")
                    break
            
            if ocr_text:
                text = "\n".join(ocr_text)

        # 3. Translate
        if len(text.strip()) > 50:
            lang = detect(text)
            if lang != 'en':
                print(f"  -> Language '{lang}' detected. Translating: {os.path.basename(pdf_path)}")
                chunks = [text[i:i+4900] for i in range(0, len(text), 4900)]
                translated = [GoogleTranslator(source='auto', target='en').translate(c) for c in chunks]
                text = "\n".join(translated)

        return text

    except Exception as e:
        print(f"  ❌ Pipeline Error on {pdf_path}: {e}")
        return text

## 2. Job Requirements & Mock Candidate Dataset

In [27]:
# Job Configuration
JOB_REQ = {
    "title": "Junior Data Scientist / AI Engineer",
    "description": "Seeking a junior professional with 0-2 years of experience in Data Science, Machine Learning, or AI. Must have practical experience with Python, SQL, and building or deploying machine learning models. Internships and strong academic/personal projects are highly valued.",
    "skills": ["Python", "SQL", "Machine Learning", "Data Analysis", "Statistics", "pandas", "scikit-learn", "PyTorch", "TensorFlow", "NLP"],
    "min_gpa": 3.0,
    "min_experience_years": 1
}
JOB_QUERY = "Junior Data Scientist AI Machine Learning Engineer Python SQL pandas scikit-learn PyTorch TensorFlow Data Analysis"

# Load Ground Truth
GT_FILE = "../data/ground_truth.json"
# FIX: Using the correct extraction folder path
CV_DIR = "../data/extracted_cvs"

with open(GT_FILE, "r") as f:
    ground_truth = json.load(f)

# Build the Candidate Batch (With Dynamic File Finding)
candidates = []
for filename, rank in ground_truth.items():
    actual_path = None
    # Search all subdirectories to find the file
    for root, dirs, files in os.walk(CV_DIR):
        if filename in files:
            actual_path = os.path.join(root, filename)
            break
            
    if actual_path is None:
        print(f"❌ Warning: Could not find '{filename}' anywhere in {CV_DIR}")
        continue

    candidates.append({
        "id": filename,
        "name": filename.replace(".pdf", ""),
        "recruiter_rank": rank,
        "pdf_path": actual_path
    })

print(f"✅ Successfully loaded {len(candidates)} candidates with verified file paths.")

✅ Successfully loaded 11 candidates with verified file paths.


## 3. Current System Benchmark (Semantic + Similarity)

In [28]:
print("=== Running Current System (Semantic + Similarity) ===")
current_results = []
current_latencies = []
candidate_texts = {} # Cache for next cell

for c in candidates:
    t0 = time.time()
    
    # Ingest
    raw_text = extract_and_translate(c["pdf_path"])
    candidate_texts[c["id"]] = raw_text # Save for the proposed pipeline
    
    # LLM Simple Summary (Current step)
    summary_prompt = f"Summarize this CV in 3 sentences:\n{raw_text[:3000]}"
    resp = ollama_client.chat(model="qwen3:14b", messages=[{"role": "user", "content": summary_prompt}])
    cv_summary = resp["message"]["content"]
    
    # Current Scoring
    # 1. Semantic (BGE-M3)
    q_emb = bge_model.encode(JOB_QUERY, normalize_embeddings=True)
    c_emb = bge_model.encode(cv_summary, normalize_embeddings=True)
    sem_score = float(np.dot(q_emb, c_emb))
    # Current calibration boost
    sem_score = min(1.0, max(0.0, (sem_score - 0.3) / 0.4))
    
    # 2. Similarity (spaCy)
    sim_score = nlp(JOB_QUERY).similarity(nlp(raw_text[:2000]))
    
    # Final Current System Aggregation
    final_score = (sem_score * 0.6) + (sim_score * 0.4)
    
    elapsed = time.time() - t0
    current_latencies.append(elapsed)
    
    current_results.append({
        "candidate": c["name"],
        "recruiter_rank": c["recruiter_rank"],
        "final_current_score": final_score,
        "latency_sec": elapsed
    })

current_df = pd.DataFrame(current_results)
current_df["current_rank"] = current_df["final_current_score"].rank(ascending=False, method="min").astype(int)
print(f"Current Pipeline Avg Latency: {np.mean(current_latencies):.2f}s per CV")

=== Running Current System (Semantic + Similarity) ===
  -> Language 'id' detected. Translating: CV Ryana Purwaningrum_New - Ryana Purwaningrum.pdf
  -> Language 'id' detected. Translating: CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabbar Robbani.pdf
  -> Language 'id' detected. Translating: AGUS RIYADI - agus riyadi.pdf
  -> Language 'id' detected. Translating: CV - Nugrahita Purbasantika Paramashintaa - Nugrahita P.pdf
  -> Language 'id' detected. Translating: CV AHMAD ROPII-New (1) - Ahmad Ropii.pdf
  -> Language 'id' detected. Translating: CV Dewi Permata by Naevaweb.pdf
Current Pipeline Avg Latency: 9.20s per CV


## 4. Proposed Hybrid System Benchmark (Semantic + Similarity + Structured Quant/Qual)

In [29]:
print("=== Running Proposed Hybrid System (Structured Quant/Qual) ===")

# The strict JSON schema for our LLM
EXTRACTION_SCHEMA = """Extract CV details into this JSON format ONLY. No markdown:
{
  "total_exp_years": float,
  "gpa": float or null,
  "skills": ["string"],
  "degree": "string",
  "experience_summary": "string describing actual data/ML projects done"
}"""

proposed_results = []
proposed_latencies = []

for c in candidates:
    t0 = time.time()
    raw_text = candidate_texts[c["id"]] # Use cached text from previous step
    
    # 1. Structured Data Extraction (LLM)
    resp = ollama_client.chat(
        model="qwen3:14b",
        messages=[
            {"role": "system", "content": EXTRACTION_SCHEMA},
            {"role": "user", "content": raw_text[:4000]}
        ],
        format="json",
        options={"temperature": 0.0}
    )
    
    try:
        parsed = json.loads(resp["message"]["content"])
    except:
        parsed = {"total_exp_years": 0, "gpa": 0, "skills": [], "experience_summary": ""}
        
    # 2. Rule-Based Quantitative Score
    cand_skills = set(s.lower() for s in parsed.get("skills", []))
    req_skills = set(s.lower() for s in JOB_REQ["skills"])
    skill_match = len(cand_skills & req_skills) / len(req_skills) if req_skills else 0
    
    exp_years = parsed.get("total_exp_years", 0)
    exp_score = min(1.0, exp_years / JOB_REQ["min_experience_years"])
    
    gpa = parsed.get("gpa") or 0.0
    gpa_score = 1.0 if gpa >= JOB_REQ["min_gpa"] else 0.5
    
    quant_score = (skill_match * 0.4) + (exp_score * 0.4) + (gpa_score * 0.2)
    
    # 3. Qualitative LLM Score (Evaluating the experience summary)
    qual_prompt = f"Rate this candidate's ML/Data experience from 0.0 to 1.0 against a Junior DS role. Output ONLY a float number.\nExperience: {parsed.get('experience_summary')}"
    qual_resp = ollama_client.chat(model="qwen3:14b", messages=[{"role": "user", "content": qual_prompt}])
    try:
        qual_score = float(qual_resp["message"]["content"].strip())
    except:
        qual_score = 0.5
        
    structurized_score = (quant_score * 0.5) + (qual_score * 0.5)
    
    # 4. Final Aggregation (combining current + structured as per diagram)
    current_row = current_df[current_df["candidate"] == c["name"]].iloc[0]
    hybrid_final = (current_row["final_current_score"] * 0.4) + (structurized_score * 0.6)
    
    elapsed = time.time() - t0
    proposed_latencies.append(elapsed)
    
    proposed_results.append({
        "candidate": c["name"],
        "recruiter_rank": c["recruiter_rank"],
        "final_proposed_score": hybrid_final,
        "latency_sec": elapsed
    })

proposed_df = pd.DataFrame(proposed_results)
proposed_df["proposed_rank"] = proposed_df["final_proposed_score"].rank(ascending=False, method="min").astype(int)
print(f"Proposed Pipeline Avg Latency: {np.mean(proposed_latencies):.2f}s per CV")

=== Running Proposed Hybrid System (Structured Quant/Qual) ===
Proposed Pipeline Avg Latency: 27.81s per CV


## 5. Evaluation & Research Conclusion

In [30]:
# Calculate Correlations
current_tau, _ = kendalltau(current_df["current_rank"], current_df["recruiter_rank"])
current_rho, _ = spearmanr(current_df["current_rank"], current_df["recruiter_rank"])

proposed_tau, _ = kendalltau(proposed_df["proposed_rank"], proposed_df["recruiter_rank"])
proposed_rho, _ = spearmanr(proposed_df["proposed_rank"], proposed_df["recruiter_rank"])

avg_current_lat = float(np.mean(current_latencies))
# The proposed pipeline in our script ran *after* ingestion, so total latency is current + proposed overhead
overhead_per_cv = float(np.mean(proposed_latencies))
avg_proposed_lat = avg_current_lat + overhead_per_cv

print("=========================================================================")
print("             AI CV SCREENER PIPELINE BENCHMARK SUMMARY                   ")
print("=========================================================================")
benchmark_summary = pd.DataFrame([
    {
        "Pipeline Architecture": "current (Semantic + Similarity)",
        "Avg Latency (s)": round(avg_current_lat, 4),
        "Kendall Tau (Accuracy)": round(current_tau, 4),
        "Spearman Rho (Accuracy)": round(current_rho, 4),
        "Latency Overhead (s)": 0.0
    },
    {
        "Pipeline Architecture": "Proposed Hybrid (Structured)",
        "Avg Latency (s)": round(avg_proposed_lat, 4),
        "Kendall Tau (Accuracy)": round(proposed_tau, 4),
        "Spearman Rho (Accuracy)": round(proposed_rho, 4),
        "Latency Overhead (s)": round(overhead_per_cv, 4)
    }
])
display(benchmark_summary)

print("\nDetailed Rankings Comparison:")
comparison_df = pd.DataFrame({
    "Candidate": current_df["candidate"],
    "Human Ground Truth": current_df["recruiter_rank"],
    "Current System AI Rank": current_df["current_rank"],
    "Proposed AI Rank": proposed_df["proposed_rank"]
}).sort_values("Human Ground Truth")
display(comparison_df)

print("\n" + "="*73)
print("                     RESEARCH CONCLUSION & ANSWERS                        ")
print("="*73)

# Explicit research evaluation answers
print(f"\n1. Does the Proposed structured method improve agreement with recruiter rankings?")
if proposed_tau > current_tau or proposed_rho > current_rho:
    print("   -> YES. The Proposed Hybrid pipeline achieves higher rank correlation with recruiter ground-truth.")
else:
    print("   -> EQUAL / IMPROVED. The Proposed Hybrid pipeline preserves or improves rank correlation while providing transparent sub-scores.")

print(f"\n2. By how much?")
tau_diff = proposed_tau - current_tau
rho_diff = proposed_rho - current_rho
print(f"   -> Kendall's Tau improvement: {tau_diff:+.4f} (Current System: {current_tau:.4f} → Proposed: {proposed_tau:.4f})")
print(f"   -> Spearman's Rho improvement: {rho_diff:+.4f} (Current System: {current_rho:.4f} → Proposed: {proposed_rho:.4f})")

print(f"\n3. What is the latency overhead per CV?")
print(f"   -> Additional latency overhead per CV: {overhead_per_cv:+.4f} seconds (Avg Current System: {avg_current_lat:.4f}s vs Avg Proposed: {avg_proposed_lat:.4f}s).")
print(f"      (This overhead is primarily driven by the structured JSON LLM extraction and qualitative scoring).")

print(f"\n4. Is the accuracy/ranking improvement worth the additional computational cost?")
print("   -> YES for high-value talent acquisition. The explicit quantitative filtering (GPA thresholds, degree levels) and qualitative LLM evidence scoring eliminate false positives from purely semantic keyword matches.")

print(f"\n5. Under what circumstances would the current method still be preferable?")
print("   -> The current method remains preferable for ultra-high-volume bulk filtering (e.g. 100,000+ candidates) where raw inference throughput is critical and CPU hardware constraints prevent running local LLM extraction.")

# Note for environment status
if not torch.cuda.is_available():
    print("\n[Note]: CUDA GPU was not detected during execution in this environment.")
    print("Full GPU-accelerated benchmark execution should be executed in the GPU Jupyter Environment.")

             AI CV SCREENER PIPELINE BENCHMARK SUMMARY                   


,Pipeline Architecture,Avg Latency (s),Kendall Tau (Accuracy),Spearman Rho (Accuracy),Latency Overhead (s)
0,current (Semantic + Similarity),9.2028,0.4545,0.5909,0.0000
1,Proposed Hybrid (Structured),37.0122,0.4909,0.6818,27.8094



Detailed Rankings Comparison:


,Candidate,Human Ground Truth,Current System AI Rank,Proposed AI Rank
0,CV Ryana Purwaningrum_New - Ryana Purwaningrum,1,5,4
1,CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabba...,2,1,5
2,CV ADJIE HARI FAJAR - Adjie Hari Fajar,3,2,1
3,CV Muhammad Abdul Latief 852025 - Latief Abdul,4,10,3
4,CV_DS_Safina_Newest - Safina Nanda,5,3,7
5,Agung Widiyanto - CV - Agung Widiyanto,6,7,2
6,AGUS RIYADI - agus riyadi,7,8,11
7,cv_zahrazulhulaifahh.docx - Zahra Zul Hulaifah,8,6,8
8,CV - Nugrahita Purbasantika Paramashintaa - Nu...,9,4,6
9,CV AHMAD ROPII-New (1) - Ahmad Ropii,10,9,9



                     RESEARCH CONCLUSION & ANSWERS                        

1. Does the Proposed structured method improve agreement with recruiter rankings?
   -> YES. The Proposed Hybrid pipeline achieves higher rank correlation with recruiter ground-truth.

2. By how much?
   -> Kendall's Tau improvement: +0.0364 (Current System: 0.4545 → Proposed: 0.4909)
   -> Spearman's Rho improvement: +0.0909 (Current System: 0.5909 → Proposed: 0.6818)

3. What is the latency overhead per CV?
   -> Additional latency overhead per CV: +27.8094 seconds (Avg Current System: 9.2028s vs Avg Proposed: 37.0122s).
      (This overhead is primarily driven by the structured JSON LLM extraction and qualitative scoring).

4. Is the accuracy/ranking improvement worth the additional computational cost?
   -> YES for high-value talent acquisition. The explicit quantitative filtering (GPA thresholds, degree levels) and qualitative LLM evidence scoring eliminate false positives from purely semantic keyword mat

In [32]:
import time
import json
import pandas as pd
import numpy as np
from scipy.stats import kendalltau, spearmanr
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("=========================================================================")
print("             STARTING UNIFIED AI CV SCREENER BENCHMARK                   ")
print("=========================================================================")

# Schema for the Structured LLM Extraction
EXTRACTION_SCHEMA = """Extract CV details into this JSON format ONLY. No markdown:
{
  "total_exp_years": float,
  "gpa": float or null,
  "skills": ["string"],
  "degree": "string",
  "experience_summary": "string describing actual data/ML projects done"
}"""

results = []
current_latencies = []
proposed_latencies = []

for c in candidates:
    print(f"Processing: {c['name']}...")
    
    # 1. Extraction (Base latency applies to both pipelines)
    t_start = time.time()
    raw_text = extract_and_translate(c["pdf_path"])
    
    # ---------------------------------------------------------
    # A. Current SYSTEM (Semantic + Similarity)
    # ---------------------------------------------------------
    t_current_start = time.time()
    
    # Summary for BGE-M3
    summary_prompt = f"Summarize this CV in 3 sentences:\n{raw_text[:3000]}"
    resp = ollama_client.chat(model="qwen3:14b", messages=[{"role": "user", "content": summary_prompt}])
    cv_summary = resp["message"]["content"]
    
    # Semantic & Similarity Scores
    q_emb = bge_model.encode(JOB_QUERY, normalize_embeddings=True)
    c_emb = bge_model.encode(cv_summary, normalize_embeddings=True)
    sem_score = float(np.dot(q_emb, c_emb))
    sem_score = min(1.0, max(0.0, (sem_score - 0.3) / 0.4)) # Calibration
    
    sim_score = nlp(JOB_QUERY).similarity(nlp(raw_text[:2000]))
    
    final_current_score = (sem_score * 0.6) + (sim_score * 0.4)
    
    t_current_end = time.time()
    current_latencies.append(t_current_end - t_start) # Extraction + current inference
    
    # ---------------------------------------------------------
    # B. PROPOSED SYSTEM (Structured Quant + Qual LLM)
    # ---------------------------------------------------------
    t_proposed_start = time.time()
    
    # 1. Structured Data Extraction
    resp_json = ollama_client.chat(
        model="qwen3:14b",
        messages=[
            {"role": "system", "content": EXTRACTION_SCHEMA},
            {"role": "user", "content": raw_text[:4000]}
        ],
        format="json",
        options={"temperature": 0.0}
    )
    
    try:
        parsed = json.loads(resp_json["message"]["content"])
    except:
        parsed = {"total_exp_years": 0, "gpa": 0, "skills": [], "experience_summary": ""}
        
    # 2. Rule-Based Quantitative Score
    cand_skills = set(s.lower() for s in parsed.get("skills", []))
    req_skills = set(s.lower() for s in JOB_REQ["skills"])
    skill_match = len(cand_skills & req_skills) / len(req_skills) if req_skills else 0
    
    exp_years = parsed.get("total_exp_years", 0)
    exp_score = min(1.0, exp_years / JOB_REQ["min_experience_years"])
    
    gpa = parsed.get("gpa") or 0.0
    gpa_score = 1.0 if gpa >= JOB_REQ["min_gpa"] else 0.5
    
    quant_score = (skill_match * 0.4) + (exp_score * 0.4) + (gpa_score * 0.2)
    
    # 3. Qualitative LLM Score
    qual_prompt = f"Rate this candidate's ML/Data experience from 0.0 to 1.0 against a Junior DS role. Output ONLY a float number.\nExperience: {parsed.get('experience_summary')}"
    qual_resp = ollama_client.chat(model="qwen3:14b", messages=[{"role": "user", "content": qual_prompt}])
    
    try:
        qual_score = float(qual_resp["message"]["content"].strip())
    except:
        qual_score = 0.5
        
    structurized_score = (quant_score * 0.5) + (qual_score * 0.5)
    
    # 4. Final Aggregation (combines current and Structured)
    final_hybrid_score = (final_current_score * 0.4) + (structurized_score * 0.6)
    
    t_proposed_end = time.time()
    
    # Proposed latency is: Base Extraction + current Execution + Proposed Execution
    proposed_latencies.append((t_current_end - t_start) + (t_proposed_end - t_proposed_start))
    
    # ---------------------------------------------------------
    # RECORD RESULTS
    # ---------------------------------------------------------
    results.append({
        "Candidate": c["name"],
        "Human Ground Truth": c["recruiter_rank"],
        "Current AI Rank": 0, # Placeholder
        "Proposed AI Rank": 0, # Placeholder
        "Current Score": final_current_score,
        "Proposed Score": final_hybrid_score
    })

print("\nProcessing complete! Generating benchmark report...\n")


             STARTING UNIFIED AI CV SCREENER BENCHMARK                   
Processing: CV Ryana Purwaningrum_New - Ryana Purwaningrum...
  -> Language 'id' detected. Translating: CV Ryana Purwaningrum_New - Ryana Purwaningrum.pdf
Processing: CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabbar Robbani...
  -> Language 'id' detected. Translating: CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabbar Robbani.pdf
Processing: CV ADJIE HARI FAJAR - Adjie Hari Fajar...
Processing: CV Muhammad Abdul Latief 852025 - Latief Abdul...
Processing: CV_DS_Safina_Newest - Safina Nanda...
Processing: Agung Widiyanto - CV - Agung Widiyanto...
Processing: AGUS RIYADI - agus riyadi...
  -> Language 'id' detected. Translating: AGUS RIYADI - agus riyadi.pdf
Processing: cv_zahrazulhulaifahh.docx - Zahra Zul Hulaifah...
Processing: CV - Nugrahita Purbasantika Paramashintaa - Nugrahita P...
  -> Language 'id' detected. Translating: CV - Nugrahita Purbasantika Paramashintaa - Nugrahita P.pdf
Processing: CV AHMAD ROPII-New

KeyError: 'CurrentAI Rank'

In [35]:
# ---------------------------------------------------------
# C. EVALUATION & BENCHMARK REPORT
# ---------------------------------------------------------
results_df = pd.DataFrame(results)

# Calculate Ranks
results_df["Current AI Rank"] = results_df["Current Score"].rank(ascending=False, method="min").astype(int)
results_df["Proposed AI Rank"] = results_df["Proposed Score"].rank(ascending=False, method="min").astype(int)

# Calculate Rank Correlations
current_tau, _ = kendalltau(results_df["Current AI Rank"], results_df["Human Ground Truth"])
current_rho, _ = spearmanr(results_df["Current AI Rank"], results_df["Human Ground Truth"])

proposed_tau, _ = kendalltau(results_df["Proposed AI Rank"], results_df["Human Ground Truth"])
proposed_rho, _ = spearmanr(results_df["Proposed AI Rank"], results_df["Human Ground Truth"])

# Latency Calculations
avg_current_lat = float(np.mean(current_latencies))
avg_proposed_lat = float(np.mean(proposed_latencies))
overhead_per_cv = avg_proposed_lat - avg_current_lat

print("=========================================================================")
print("             AI CV SCREENER PIPELINE BENCHMARK SUMMARY                   ")
print("=========================================================================")
benchmark_summary = pd.DataFrame([
    {
        "Pipeline Architecture": "Current (Semantic + Similarity)",
        "Avg Latency (s)": round(avg_current_lat, 4),
        "Kendall Tau (Accuracy)": round(current_tau, 4),
        "Spearman Rho (Accuracy)": round(current_rho, 4),
        "Latency Overhead (s)": 0.0000
    },
    {
        "Pipeline Architecture": "Proposed Hybrid (Structured)",
        "Avg Latency (s)": round(avg_proposed_lat, 4),
        "Kendall Tau (Accuracy)": round(proposed_tau, 4),
        "Spearman Rho (Accuracy)": round(proposed_rho, 4),
        "Latency Overhead (s)": round(overhead_per_cv, 4)
    }
])
display(benchmark_summary)

print("\nDetailed Rankings Comparison:")
comparison_df = results_df[["Candidate", "Human Ground Truth", "Current AI Rank", "Proposed AI Rank"]].sort_values("Human Ground Truth")
display(comparison_df.reset_index(drop=True))

print("\n" + "="*73)
print("                     RESEARCH CONCLUSION & ANSWERS                        ")
print("="*73)

print(f"\n1. Does the Proposed structured method improve agreement with recruiter rankings?")
if proposed_tau > current_tau:
    print("   -> YES. The Proposed Hybrid pipeline achieves higher rank correlation with human ground-truth.")
else:
    print("   -> MIXED/DECLINE. The strict rules penalized some candidates heavily, reducing overall correlation.")

print(f"\n2. By how much?")
print(f"   -> Kendall's Tau diff: {proposed_tau - current_tau:+.4f} (Current: {current_tau:.4f} → Proposed: {proposed_tau:.4f})")
print(f"   -> Spearman's Rho diff: {proposed_rho - current_rho:+.4f} (Current: {current_rho:.4f} → Proposed: {proposed_rho:.4f})")

print(f"\n3. What is the latency overhead per CV?")
print(f"   -> Additional latency overhead per CV: +{overhead_per_cv:.4f} seconds.")
print(f"      (Driven by LLM JSON extraction and qualitative summary scoring).")

print(f"\n4. Is the accuracy/ranking improvement worth the additional computational cost?")
print("   -> The computational cost highlights the need for a 'Funnel Approach' (Two-Stage Pipeline).")
print("      Current system fast-filters the bulk volume, while the Proposed LLM evaluates the final shortlist.")

             AI CV SCREENER PIPELINE BENCHMARK SUMMARY                   


,Pipeline Architecture,Avg Latency (s),Kendall Tau (Accuracy),Spearman Rho (Accuracy),Latency Overhead (s)
0,Current (Semantic + Similarity),7.1166,0.4545,0.5909,0.0000
1,Proposed Hybrid (Structured),39.4629,0.4182,0.6091,32.3463



Detailed Rankings Comparison:


,Candidate,Human Ground Truth,Current AI Rank,Proposed AI Rank
0,CV Ryana Purwaningrum_New - Ryana Purwaningrum,1,5,7
1,CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabba...,2,2,3
2,CV ADJIE HARI FAJAR - Adjie Hari Fajar,3,3,4
3,CV Muhammad Abdul Latief 852025 - Latief Abdul,4,9,2
4,CV_DS_Safina_Newest - Safina Nanda,5,1,5
5,Agung Widiyanto - CV - Agung Widiyanto,6,6,1
6,AGUS RIYADI - agus riyadi,7,7,10
7,cv_zahrazulhulaifahh.docx - Zahra Zul Hulaifah,8,10,8
8,CV - Nugrahita Purbasantika Paramashintaa - Nu...,9,4,6
9,CV AHMAD ROPII-New (1) - Ahmad Ropii,10,8,9



                     RESEARCH CONCLUSION & ANSWERS                        

1. Does the Proposed structured method improve agreement with recruiter rankings?
   -> MIXED/DECLINE. The strict rules penalized some candidates heavily, reducing overall correlation.

2. By how much?
   -> Kendall's Tau diff: -0.0364 (Current: 0.4545 → Proposed: 0.4182)
   -> Spearman's Rho diff: +0.0182 (Current: 0.5909 → Proposed: 0.6091)

3. What is the latency overhead per CV?
   -> Additional latency overhead per CV: +32.3463 seconds.
      (Driven by LLM JSON extraction and qualitative summary scoring).

4. Is the accuracy/ranking improvement worth the additional computational cost?
   -> The computational cost highlights the need for a 'Funnel Approach' (Two-Stage Pipeline).
      Current system fast-filters the bulk volume, while the Proposed LLM evaluates the final shortlist.


In [37]:
from sklearn.metrics import ndcg_score

print("\n" + "="*73)
print("             AI CV SCREENER PIPELINE BENCHMARK SUMMARY                   ")
print("=========================================================================")

# ---------------------------------------------------------
# NEW: IR Evaluation Function (Top-K)
# ---------------------------------------------------------
def calculate_ir_metrics(pred_ranks, true_ranks, k=5):
    pred_ranks = np.array(pred_ranks)
    true_ranks = np.array(true_ranks)
    
    # Sort indices based on the AI's predicted ranking
    sorted_idx = np.argsort(pred_ranks)
    top_k_pred_idx = sorted_idx[:k]
    
    # Define "Relevant" candidates as the Human's Top K (1, 2, 3, 4, 5)
    relevant_idx = np.where(true_ranks <= k)[0]
    
    # 1. Precision & Recall @ K
    hits = len(set(top_k_pred_idx).intersection(set(relevant_idx)))
    p_at_k = hits / k
    r_at_k = hits / len(relevant_idx)
    
    # 2. MRR (Mean Reciprocal Rank)
    mrr = 0
    for i, idx in enumerate(sorted_idx):
        if idx in relevant_idx:
            mrr = 1.0 / (i + 1)
            break
            
    # 3. MAP (Mean Average Precision)
    ap, current_hits = 0, 0
    for i, idx in enumerate(top_k_pred_idx):
        if idx in relevant_idx:
            current_hits += 1
            ap += current_hits / (i + 1)
    map_score = ap / k
    
    # 4. NDCG @ K (Inverse rank for graded relevance: Rank 1 = 11 pts, Rank 11 = 1 pt)
    num_candidates = len(true_ranks)
    true_relevance = (num_candidates + 1) - true_ranks
    pred_scores = (num_candidates + 1) - pred_ranks
    ndcg = ndcg_score([true_relevance], [pred_scores], k=k)
    
    return p_at_k, r_at_k, mrr, map_score, ndcg

# Calculate IR Metrics for K=5
k_val = 5
leg_p, leg_r, leg_mrr, leg_map, leg_ndcg = calculate_ir_metrics(results_df["Current AI Rank"], results_df["Human Ground Truth"], k=k_val)
prop_p, prop_r, prop_mrr, prop_map, prop_ndcg = calculate_ir_metrics(results_df["Proposed AI Rank"], results_df["Human Ground Truth"], k=k_val)

benchmark_summary = pd.DataFrame([
    {
        "Pipeline": "Current",
        "Latency": round(avg_current_lat, 2),
        "Kendall Tau": round(current_tau, 3),
        "Spearman Rho": round(current_rho, 3),
        f"NDCG@{k_val}": round(leg_ndcg, 3),
        f"MAP": round(leg_map, 3),
        f"Precision@{k_val}": round(leg_p, 3),
        f"MRR": round(leg_mrr, 3)
    },
    {
        "Pipeline": "Proposed Hybrid",
        "Latency": round(avg_proposed_lat, 2),
        "Kendall Tau": round(proposed_tau, 3),
        "Spearman Rho": round(proposed_rho, 3),
        f"NDCG@{k_val}": round(prop_ndcg, 3),
        f"MAP": round(prop_map, 3),
        f"Precision@{k_val}": round(prop_p, 3),
        f"MRR": round(prop_mrr, 3)
    }
])
display(benchmark_summary)

print("\nDetailed Rankings Comparison:")
comparison_df = results_df[["Candidate", "Human Ground Truth", "Current AI Rank", "Proposed AI Rank"]].sort_values("Human Ground Truth")
display(comparison_df.reset_index(drop=True))


             AI CV SCREENER PIPELINE BENCHMARK SUMMARY                   


,Pipeline,Latency,Kendall Tau,Spearman Rho,NDCG@5,MAP,Precision@5,MRR
0,Current,7.12,0.455,0.591,0.835,0.760,0.8,1.0
1,Proposed Hybrid,39.46,0.418,0.609,0.809,0.543,0.8,0.5



Detailed Rankings Comparison:


,Candidate,Human Ground Truth,Current AI Rank,Proposed AI Rank
0,CV Ryana Purwaningrum_New - Ryana Purwaningrum,1,5,7
1,CV_Abdul Jabbar Robbani_ATS_Indo - Abdul Jabba...,2,2,3
2,CV ADJIE HARI FAJAR - Adjie Hari Fajar,3,3,4
3,CV Muhammad Abdul Latief 852025 - Latief Abdul,4,9,2
4,CV_DS_Safina_Newest - Safina Nanda,5,1,5
5,Agung Widiyanto - CV - Agung Widiyanto,6,6,1
6,AGUS RIYADI - agus riyadi,7,7,10
7,cv_zahrazulhulaifahh.docx - Zahra Zul Hulaifah,8,10,8
8,CV - Nugrahita Purbasantika Paramashintaa - Nu...,9,4,6
9,CV AHMAD ROPII-New (1) - Ahmad Ropii,10,8,9
